In [5]:
import findspark
findspark.init()

In [6]:
# Import the needed Python modules
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from datetime import datetime

In [7]:
# Create a SparkSession
spark = SparkSession.builder \
    .appName("CreateDataFrameWithSchema") \
    .getOrCreate()

In [8]:
import inspect
print(display)
print(inspect.getmodule(display))

<function display at 0x00000168806F0B80>
<module 'IPython.core.display_functions' from 'c:\\Users\\user\\SoftwareProjects\\Coursera\\IBM_DE\\ibm-data-engineering\\venv312_ibmde\\Lib\\site-packages\\IPython\\core\\display_functions.py'>


In [2]:
!pip list

Package                 Version
----------------------- -----------
asttokens               3.0.2
colorama                0.4.6
comm                    0.2.3
debugpy                 1.8.22
executing               2.2.1
findspark               2.0.1
ipykernel               7.4.0
ipython                 9.17.1
ipython-genutils        0.2.0
ipython_pygments_lexers 1.1.1
ipython-sql             0.5.0
jedi                    0.20.0
jupyter_client          8.10.0
jupyter_core            5.9.1
matplotlib-inline       0.2.2
nest-asyncio2           1.7.3
packaging               26.3
parso                   0.8.7
pip                     26.2.1
platformdirs            4.12.2
prettytable             3.18.0
prompt_toolkit          3.0.53
psutil                  7.2.2
pure_eval               0.2.4
Pygments                2.21.0
python-dateutil         2.9.0.post0
pyzmq                   27.2.0
six                     1.17.0
SQLAlchemy              2.1.1
sqlparse                0.6.0
stack-data      

In [10]:
# Create a sample DataFrame with book inventory
data = [
    (101, "The Great Gatsby", 10),
    (102, "To Kill a Mockingbird", 5),
    (103, "1984", 15)
]
columns = ["book_id", "title", "quantity"]

df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("inventory_tmpvw")
# Write the DataFrame to a Delta table in your Lakehouse
# df.write.format("delta").mode("overwrite").saveAsTable("inventory")

# Display the table to confirm it was created
df.show()
# display(spark.sql("SELECT * FROM inventory_tmpvw"))

+-------+--------------------+--------+
|book_id|               title|quantity|
+-------+--------------------+--------+
|    101|    The Great Gatsby|      10|
|    102|To Kill a Mocking...|       5|
|    103|                1984|      15|
+-------+--------------------+--------+



In [ ]:
from pyspark.sql.functions import col, when

# Perform a successful update (durability demo)
print("Performing a successful sale of 'To Kill a Mockingbird'...")

# Read the latest version of the table
#inventory_table = spark.table("inventory")

# Decrease the quantity for book_id 102
updated_df = (
inventory_table
.withColumn("quantity",
when(col("book_id") == 102, col("quantity") - 1)
.otherwise(col("quantity")))
)

# Write the change as a single overwrite to the same table (creates a new table version)
updated_df.write.format("delta").mode("overwrite").saveAsTable("inventory")

print("Update successful! Latest table state:")
display(spark.table("inventory"))


In [ ]:
# Use Time Travel to query the original version of the table
print("Using Time Travel to see the table before the last sale...")

df_version_0 = spark.read.format("delta").option("versionAsOf", 0).table("inventory")

display(df_version_0)

In [ ]:
# Show the ordered commit history for the table
history_df = spark.sql("DESCRIBE HISTORY inventory")
display(history_df)

In [ ]:
%%sql
-- Accounts: current balances
CREATE TABLE IF NOT EXISTS accounts (
  account_id INT,
  balance DECIMAL(18,2)
);

-- Ledger: immutable debit/credit journal
CREATE TABLE IF NOT EXISTS ledger (
  entry_id BIGINT,
  ts TIMESTAMP,
  from_acct INT,
  to_acct INT,
  amount DECIMAL(18,2),   -- positive amount = transfer magnitude
  note STRING
);

-- Seed data (idempotent-ish: upsert or ensure only once in your lab)
DELETE FROM accounts;
INSERT INTO accounts (account_id, balance) VALUES
(1001, 1000.00),
(2002,  250.00);

-- Clean slate for ledger
DELETE FROM ledger;

In [ ]:
%%sql
-- Build the transfer deltas as a derived set:
-- 1001: -200.00, 2002: +200.00
MERGE INTO accounts AS t
USING (
  SELECT 1001 AS account_id, CAST(-200.00 AS DECIMAL(18,2)) AS delta UNION ALL
  SELECT 2002,  CAST( 200.00 AS DECIMAL(18,2))
) AS s
ON t.account_id = s.account_id
WHEN MATCHED THEN UPDATE SET t.balance = t.balance + s.delta;

In [ ]:
%%sql
SELECT * FROM accounts ORDER BY account_id;

-- Invariant check: sum of deltas across all affected accounts should be 0
-- (Here we re-check current pair)
SELECT SUM(CASE WHEN account_id IN (1001,2002) THEN balance END)
FROM accounts; -- not strictly zero; instead, check two-row delta sum if you logged pre-state